# Advanced RAG with LangChain, Docling & ChromaDB

This notebook builds a **production-grade RAG pipeline** using the same four stages as the simple
notebook, but with real libraries used in production systems.

| Dimension | Simple Notebook | This Notebook |
|---|---|---|
| **Documents** | Hardcoded strings | Markdown files parsed by **Docling** |
| **Chunking** | Naive word-window | Smart `RecursiveCharacterTextSplitter` |
| **Vector store** | In-memory NumPy | **ChromaDB** — persistent on disk |
| **Pipeline** | Manual function calls | **LangChain LCEL** declarative chains |
| **Memory** | Stateless | **Conversation history** for follow-ups |
| **Sources** | None | Source attribution per answer |
| **Filtering** | None | **Metadata filter** by document |

Run cells top-to-bottom. Each section explains *why* the design decision matters.

In [ ]:
%pip install -q \
    langchain \
    langchain-openai \
    langchain-chroma \
    langchain-huggingface \
    langchain-docling \
    langchain-text-splitters \
    chromadb \
    docling \
    sentence-transformers

In [ ]:
import os
import getpass
import textwrap
import shutil
from pathlib import Path

from langchain_docling import DoclingLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain.chains import create_retrieval_chain, create_history_aware_retriever
from langchain.chains.combine_documents import create_stuff_documents_chain

---
## Configuration

The same environment variables as the simple notebook work here.
For a local LiteLLM proxy set `LLM_BASE_URL` and `LLM_MODEL`;
for OpenAI leave both empty and set `OPENAI_API_KEY`.

In [ ]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"   # same model as the simple notebook for comparison
LLM_MODEL       = os.getenv("LLM_MODEL", "gpt-4o-mini")
LLM_BASE_URL    = os.getenv("LLM_BASE_URL", "")  # e.g. http://localhost:4000
CHROMA_DIR      = "./chroma_db"         # ChromaDB persists here across runs
DOCS_DIR        = Path("./docs")
CHUNK_SIZE      = 1000                  # characters (not words — splitter is char-based)
CHUNK_OVERLAP   = 150
TOP_K           = 4

print(f"LLM          : {LLM_MODEL!r}")
print(f"API base     : {LLM_BASE_URL or '(provider default)'}")
print(f"Embeddings   : {EMBEDDING_MODEL!r}")
print(f"ChromaDB dir : {CHROMA_DIR!r}")

In [ ]:
# Set the API key for your provider.
# For a keyless local backend (Ollama, LiteLLM with no auth), skip this cell.
if not os.getenv("OPENAI_API_KEY"):
    if LLM_BASE_URL:
        os.environ["OPENAI_API_KEY"] = "not-needed"
    else:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

---
## Stage 1 · Document Loading with Docling

**Why Docling?**  Real documents are not clean strings — they have headings, tables,
code blocks, footnotes, and layout artifacts. Docling understands document *structure*
and converts any supported format (PDF, DOCX, HTML, Markdown, PPTX) into a normalised
internal representation, then exports clean Markdown.

This matters because:
- A table extracted by Docling stays a table (not mangled column noise)
- Headings become `##` markers that the text splitter later uses as natural break points
- Page headers / footers / watermarks in PDFs are identified and stripped

Here we create four Markdown files that cover topics **beyond** the simple notebook.
In a real system you'd point `DoclingLoader` at a folder of PDFs or a SharePoint URL.

In [ ]:
DOCS_DIR.mkdir(exist_ok=True)

documents = {
    "transformer_architecture.md": """
# The Transformer Architecture

## Overview

The Transformer (Vaswani et al., 2017) is the foundation of every modern large language model.
Unlike RNNs it processes all tokens in parallel, making GPU training dramatically faster.

## Self-Attention

Self-attention lets every token attend to every other token simultaneously.
For "The trophy didn't fit because it was too big", attention identifies that
"it" refers to "trophy" by attending to both words in one step.

Three learned projections drive attention:
- **Query (Q)** — what this token is searching for
- **Key (K)**   — what this token advertises to others
- **Value (V)** — the content aggregated by matching queries

    Attention(Q, K, V) = softmax( Q * K^T / sqrt(d_k) ) * V

Dividing by sqrt(d_k) prevents dot products from exploding in high dimensions,
which would push softmax into near-zero-gradient saturation.

## Multi-Head Attention

Rather than a single attention function, Transformers run H heads in parallel.
Each head projects Q, K, V differently and can specialise: one head might track
syntactic agreement, another coreference, another positional distance.

    MultiHead(Q, K, V) = Concat(head_1, ..., head_H) * W_O

GPT-3 uses 96 heads across 96 layers. The diversity of head specialisations is
what gives large Transformers their broad reasoning capability.

## Positional Encoding

Because Transformers process tokens in parallel they have no inherent sense of order.
Position is injected as sinusoidal encodings added to the token embeddings:

    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

Modern models use Rotary Position Embeddings (RoPE) or ALiBi, which generalise
better to sequences longer than the training context.

## Feed-Forward Sub-layer and Residuals

After attention each position passes independently through a two-layer FFN:

    FFN(x) = max(0, x*W1 + b1) * W2 + b2

FFN layers store most of the factual knowledge learned during pretraining.
Both attention and FFN sub-layers use residual connections and layer normalisation
to keep gradients stable across hundreds of stacked layers.
""",

    "rag_systems.md": """
# Retrieval-Augmented Generation (RAG)

## The Problem RAG Solves

LLMs are trained on a fixed data snapshot. They cannot:
1. Access private documents or recent events after their training cutoff.
2. Retrieve precise facts reliably — they hallucinate plausible-sounding alternatives.

RAG fixes this by making retrieval explicit, separating **what the model knows**
from **what it can access**.

## Pipeline

**Offline (once):** load → parse → chunk → embed → store in vector DB

**Online (per query):** embed query → top-K retrieval → inject into prompt → generate

## Advanced Techniques

### Hybrid Search
Pure vector search misses exact matches (product codes, names). Hybrid search
combines dense retrieval with sparse BM25 keyword search, merging results
via Reciprocal Rank Fusion (RRF).

### HyDE (Hypothetical Document Embeddings)
Embed a hypothetical LLM-generated answer instead of the raw query.
Bridges the vocabulary gap between short questions and long passages.

### Parent-Child Chunking
Retrieve small precise child chunks for accurate matching, but inject their
larger parent chunks into the prompt for richer context.

### Reranking with Cross-Encoders
Bi-encoders (embedding models) are fast but approximate scorers. A cross-encoder
reads query and document jointly and is far more accurate. Use it to rerank
top-50 bi-encoder hits down to top-5 before generating.

### Maximal Marginal Relevance (MMR)
Instead of the top-K most similar chunks, MMR selects chunks that are
both relevant to the query AND diverse from each other. Prevents retrieving
the same sentence repeated across overlapping chunks.

## Evaluation

| Metric | Measures |
|---|---|
| Faithfulness | Are claims supported by retrieved context? |
| Answer Relevance | Does the answer address the question? |
| Context Precision | What fraction of retrieved chunks were actually used? |
| Context Recall | Were all relevant chunks retrieved? |

Tools: RAGAS, TruLens, DeepEval.
""",

    "vector_databases.md": """
# Vector Databases

## What is a Vector Database?

A vector database stores high-dimensional embeddings and supports efficient
similarity search. Traditional databases match by exact value; vector databases
match by approximate semantic meaning.

Given a query vector it returns the nearest neighbours — documents whose
embeddings point in a similar direction in the high-dimensional space.

## ChromaDB

ChromaDB is an open-source developer-focused vector database.

Key features:
- **Persistent storage** — SQLite + Parquet on disk; no separate server needed for dev
- **Metadata filtering** — filter by any document attribute before or after ANN search
- **Collections** — isolated namespaces for different document sets
- **Client-server mode** — HTTP server for multi-process access in production

## Approximate Nearest Neighbor (ANN) Search

Exact search compares the query against every vector — O(N) cost that doesn't scale.
ANN algorithms trade a small accuracy loss for massive speed gains:

- **HNSW** (Hierarchical Navigable Small World) — graph-based, excellent speed/accuracy
  tradeoff. Used by ChromaDB, Qdrant, Weaviate.
- **IVF** (Inverted File Index) — partition the space into clusters, search only nearby
  clusters. Used by FAISS.
- **ScaNN** — Google's production ANN library.

## Comparison

| Database | Scale | Highlights |
|---|---|---|
| ChromaDB | <10M vectors | Simple API, great for development |
| FAISS | 100M+ vectors | Library (no server), maximum control |
| Qdrant | 1B+ vectors | Rust, fast, rich payload filtering |
| Pinecone | 1B+ vectors | Fully managed, zero ops |
| Weaviate | 1B+ vectors | GraphQL API, multimodal |
| pgvector | <50M vectors | SQL + vectors in one PostgreSQL DB |
""",

    "fine_tuning_vs_rag.md": """
# Fine-Tuning vs RAG

## Core Trade-off

Both approaches extend LLM capabilities beyond training data, through fundamentally
different mechanisms:

- **Fine-tuning** — bakes new knowledge into weights through additional training
- **RAG** — retrieves knowledge at inference time and injects it into the prompt

## When to Use RAG

1. **Knowledge changes frequently** — legal docs, product catalogs, news. Update the
   vector store without retraining.
2. **Large knowledge base** — thousands of documents are cheaper to embed than to fine-tune on.
3. **Source attribution required** — RAG naturally returns the source chunks.
4. **Exact recall matters** — fine-tuning blends and corrupts facts; RAG retrieves verbatim.
5. **Low budget** — no GPU training cost, just inference + vector search.

## When to Use Fine-Tuning

1. **New capabilities, not just knowledge** — a new output format, reasoning style, or
   domain-specific behavior requires fine-tuning, not retrieval.
2. **Latency is critical** — no retrieval step means faster responses.
3. **Privacy** — no documents leave your system at query time.
4. **Small, stable knowledge** — if the target knowledge fits in context and rarely changes.

## LoRA and QLoRA

Full fine-tuning updates all parameters — expensive and risks catastrophic forgetting.
Parameter-Efficient Fine-Tuning (PEFT) updates only a small fraction:

**LoRA** injects trainable low-rank matrices into attention layers:

    W' = W + B*A   where rank(B*A) = r << min(d, k)

Reduces trainable parameters by 10,000x with comparable final performance.

**QLoRA** combines 4-bit NF4 quantization with LoRA, enabling fine-tuning of
a 70B parameter model on a single 48 GB GPU.

## Using Both Together

The best production systems often combine them:

1. **Fine-tune for behavior** — teach the model structured outputs, tone, guardrails.
2. **RAG for knowledge** — use the fine-tuned model as the generator with live retrieval.

This separates concerns cleanly: fine-tuning handles *how to respond*,
RAG handles *what to say*.

| Dimension | RAG | Fine-Tuning |
|---|---|---|
| Update cost | Low (re-embed docs) | High (retrain) |
| Knowledge freshness | Real-time | Stale between runs |
| Hallucination risk | Lower (grounded) | Higher |
| Latency | Higher (retrieval) | Lower |
| Explainability | High (sources shown) | Low |
"""
}

for filename, content in documents.items():
    (DOCS_DIR / filename).write_text(content.strip())

print(f"Created {len(documents)} documents in {DOCS_DIR}/")
for f in sorted(DOCS_DIR.iterdir()):
    print(f"  {f.name:35s} {f.stat().st_size:,} bytes")

In [ ]:
# DoclingLoader parses every file in the directory.
# In production, point file_path at a folder of PDFs, a single PDF URL,
# or a SharePoint / S3 path via the appropriate Docling backend.
loader = DoclingLoader(file_path=str(DOCS_DIR))
raw_docs = loader.load()

print(f"Loaded {len(raw_docs)} document(s)\n")
for doc in raw_docs:
    src = Path(doc.metadata.get("source", "?")).name
    print(f"  {src:40s} {len(doc.page_content):,} chars")

# Docling adds structured metadata — useful for advanced filtering
print(f"\nMetadata keys on first doc: {list(raw_docs[0].metadata.keys())}")

---
## Stage 1b · Smart Chunking

The simple notebook splits on word count with a fixed window.
`RecursiveCharacterTextSplitter` tries a **hierarchy of separators** in order:

```
separators = ["\n## ", "\n### ", "\n\n", "\n", " ", ""]
```

It tries the first separator that produces chunks ≤ `chunk_size` characters.
This means section boundaries (`## Heading`) are respected before paragraph breaks,
which in turn are respected before line breaks — **natural text units stay together**.

The `chunk_overlap` carries a window of text from the previous chunk so sentences
that span a boundary appear in both chunks, preventing silent information loss.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n## ", "\n### ", "\n\n", "\n", " ", ""],
)

chunks = splitter.split_documents(raw_docs)

sizes = [len(c.page_content) for c in chunks]
print(f"{len(raw_docs)} documents  →  {len(chunks)} chunks")
print(f"Chunk size  :  min={min(sizes)}, max={max(sizes)}, avg={sum(sizes)//len(sizes)} chars")

print("\nFirst 2 chunks:\n")
for chunk in chunks[:2]:
    src = Path(chunk.metadata.get("source", "")).name
    print(f"[source: {src}]")
    print(textwrap.fill(chunk.page_content[:250], width=80))
    print()

---
## Stage 2 · ChromaDB — Persistent Vector Store

The simple notebook stores embeddings in a NumPy matrix in memory — fast to build,
gone when the kernel restarts.

ChromaDB writes vectors to disk (`CHROMA_DIR`) so the index survives across sessions.
On the second run you can skip re-embedding entirely (see the persistence section at
the end of this notebook).

Additional features over a plain NumPy index:
- **Metadata storage** — every chunk carries its source, page number, etc.
- **Metadata filtering** — retrieve only from a specific document or date range
- **Collections** — multiple independent vector spaces in one store
- **`update` / `delete`** — add or remove documents without full re-indexing

In [ ]:
# Clear any previous run so the cell is idempotent
if Path(CHROMA_DIR).exists():
    shutil.rmtree(CHROMA_DIR)

print(f"Loading embedding model '{EMBEDDING_MODEL}' ...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print(f"Building ChromaDB at '{CHROMA_DIR}' ...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name="rag_advanced",
)

count = vectorstore._collection.count()
print(f"\nIndexed {count} chunks  →  persisted to '{CHROMA_DIR}/'")

---
## Stage 3 · Retrieval with MMR

The simple notebook uses plain cosine similarity (`top-K by score`).

**Maximal Marginal Relevance (MMR)** selects chunks that are simultaneously:
1. Relevant to the query (high similarity)
2. Diverse from the already-selected chunks (low pairwise similarity)

```
MMR = argmax [ lambda * sim(chunk, query) - (1-lambda) * max sim(chunk, selected) ]
```

`lambda_mult=1.0` → pure similarity (same as simple notebook)  
`lambda_mult=0.0` → maximum diversity  
`lambda_mult=0.7` → good balance (default here)

MMR prevents the retriever from returning five overlapping chunks that all say the
same thing when the document has repeated phrasing.

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": TOP_K,         # final number of chunks returned
        "fetch_k": 20,      # candidate pool considered before MMR re-ranking
        "lambda_mult": 0.7,
    },
)

test_query = "How does the attention mechanism work?"
results = retriever.invoke(test_query)

print(f"Query: {test_query!r}")
print(f"Retrieved {len(results)} chunks:\n")
for i, doc in enumerate(results, 1):
    src = Path(doc.metadata.get("source", "")).name
    print(f"[{i}] {src}")
    print(textwrap.fill(doc.page_content[:200], width=80))
    print()

---
## Stage 4 · RAG Chain with LCEL

LangChain Expression Language (LCEL) lets you **compose** pipeline steps declaratively.
The chain below is equivalent to the manual pipeline in the simple notebook, but:
- Steps are composable and swappable (swap the LLM, swap the retriever, add a reranker)
- Streaming is automatic — call `.stream()` instead of `.invoke()` for token-by-token output
- The return dict includes `context` (the actual `Document` objects) for source attribution

```
rag_chain:
  input (question)
    ↓
  retriever        → top-K Document objects
    ↓
  stuff_chain      → format docs into prompt, call LLM
    ↓
  {answer, context, input}
```

In [ ]:
llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LLM_BASE_URL or None,
    temperature=0,
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful teaching assistant. Answer the student's question "
     "using ONLY the context passages below. If the context is insufficient "
     "to answer, say so clearly — do not guess.\n\n"
     "CONTEXT:\n{context}"),
    ("human", "{input}"),
])

# create_stuff_documents_chain formats the retrieved docs into the {context} slot
combine_chain = create_stuff_documents_chain(llm, qa_prompt)

# create_retrieval_chain wires retriever → combine_chain and returns
# {input, context (list[Document]), answer}
rag_chain = create_retrieval_chain(retriever, combine_chain)

In [ ]:
def ask(question):
    result = rag_chain.invoke({"input": question})
    print(f"Q: {question}")
    print(f"\nA: {textwrap.fill(result['answer'], width=72)}")
    print("\nSources:")
    seen = set()
    for doc in result["context"]:
        src = Path(doc.metadata.get("source", "")).name
        if src not in seen:
            print(f"  - {src}")
            seen.add(src)
    print()


ask("What is multi-head attention and why does it use multiple heads?")
ask("What is the difference between RAG and fine-tuning?")
ask("When should I use ChromaDB vs Qdrant?")

---
## Conversational RAG — Follow-up Questions

The simple notebook handles each question in isolation.
In a real chat interface users ask follow-ups:

> User: What is LoRA?  
> Bot: LoRA injects low-rank matrices …  
> User: **How much memory does it save?**  ← "it" = LoRA, but the retriever doesn't know that

A **history-aware retriever** solves this by asking the LLM to rewrite the follow-up
into a standalone search query before hitting the vector store:

```
"How much memory does it save?"  +  [LoRA context from history]
           ↓  (LLM rewrites)
"How much GPU memory does LoRA fine-tuning save compared to full fine-tuning?"
           ↓
  vector retrieval  →  combine_chain  →  answer
```

This adds one extra LLM call per turn but makes retrieval dramatically more accurate
for ambiguous follow-up questions.

In [ ]:
condense_prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
    ("human",
     "Given the conversation above, rewrite the last question as a complete "
     "standalone search query. Return ONLY the rewritten query."),
])

history_aware_retriever = create_history_aware_retriever(
    llm, retriever, condense_prompt
)

conv_qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful teaching assistant. Answer using ONLY the provided "
     "context. If the context is insufficient, say so.\n\nCONTEXT:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

conv_chain = create_retrieval_chain(
    history_aware_retriever,
    create_stuff_documents_chain(llm, conv_qa_prompt),
)

In [ ]:
chat_history = []

def chat(question):
    global chat_history
    result = conv_chain.invoke({"input": question, "chat_history": chat_history})
    answer = result["answer"]
    chat_history.extend([
        HumanMessage(content=question),
        AIMessage(content=answer),
    ])
    print(f"User : {question}")
    print(f"Bot  : {textwrap.fill(answer, width=72, subsequent_indent='       ')}")
    print()


chat("What is LoRA fine-tuning?")
chat("How does it compare to full fine-tuning in GPU memory usage?")  # "it" = LoRA
chat("And what about QLoRA — how does that extend the approach?")     # follow-up

---
## Metadata Filtering

ChromaDB stores the `source` path alongside each chunk.
You can pass a `filter` dict to restrict retrieval to a specific document —
useful when your vector store has hundreds of sources and the user has already
navigated to a specific one.

ChromaDB's filter operators include `$eq`, `$ne`, `$in`, `$contains`, and more,
supporting complex boolean combinations with `$and` / `$or`.

In [ ]:
def ask_filtered(question, filename):
    """Retrieve chunks only from a specific source file."""
    source_path = str(DOCS_DIR / filename)
    filtered_retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": TOP_K,
            "filter": {"source": source_path},
        },
    )
    docs = filtered_retriever.invoke(question)
    print(f"Q (filtered to {filename}): {question}")
    print(f"Chunks retrieved: {len(docs)}\n")
    for i, doc in enumerate(docs, 1):
        print(f"[{i}] {textwrap.fill(doc.page_content[:180], width=80)}")
        print()


# Without filter: might pull from transformer_architecture.md too
# With filter: only hits vector_databases.md
ask_filtered("What indexing algorithms are used for fast search?", "vector_databases.md")

---
## ChromaDB Persistence — Reload Without Re-embedding

This is one of the most practical advantages of a persistent vector store.
Embedding a large corpus can take minutes or hours. With ChromaDB you do it
once; subsequent runs just memory-map the existing index.

The `Chroma` constructor with `persist_directory` loads an existing collection
without calling the embedding model at all.

In [ ]:
# Simulate a fresh session: reload the store from disk with no re-embedding
reloaded_store = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings,
    collection_name="rag_advanced",
)

count = reloaded_store._collection.count()
print(f"Reloaded ChromaDB: {count} chunks from '{CHROMA_DIR}'")
print("No re-embedding needed — vectors are already on disk.\n")

# Sanity-check that retrieval works identically
results = reloaded_store.similarity_search("ANN search algorithms", k=2)
print("Top-2 results for 'ANN search algorithms':")
for r in results:
    src = Path(r.metadata.get("source", "")).name
    print(f"  [{src}] {r.page_content[:100].strip()}...")

---
## What's Next?

| Upgrade | How |
|---|---|
| **Real PDFs** | Point `DoclingLoader` at a local PDF path or arXiv URL |
| **Hybrid search** | Add `BM25Retriever` + `EnsembleRetriever` from `langchain_community` |
| **Reranking** | Wrap with `FlashrankRerank` or a cross-encoder from `sentence-transformers` |
| **Better embeddings** | Swap `all-MiniLM-L6-v2` for `all-mpnet-base-v2` or `nomic-embed-text` |
| **Streaming** | Replace `.invoke()` with `.stream()` for token-by-token output |
| **Evaluation** | Add `ragas` and run faithfulness + context recall scoring |
| **Production store** | Point `Chroma` at a running ChromaDB server or swap for Qdrant |